<img src="images/econ140R_logo.png" width="200" />

<h1>ECON 140R Class 15</h1>

<h2>The Fulton Fish Market in the Wooldridge data repository</h2>

The `fish` dataset in the `wooldridge` [package](https://cran.r-project.org/web/packages/wooldridge/) is useful for understanding instrumental variables "in the wild," outside the lab settings of a randomized lottery like the Knowledge is Power Program (KIPP) showcased in <i>Mastering Metrics</i> Chapter 3. 

<h2>Learning objectives:</h2>

1. Load part of the Wooldridge data repository in __R__
2. Browse the PDF documentation as needed
3. See some useful __R__ code
4. OLS with `lm` and at the very end, some IV with `ivreg`

Let's start by installing and adding the package to our library.

In [ ]:
install.packages("wooldridge")

In [ ]:
library("wooldridge")

## Browsing the Contents

With 115 datasets in the `wooldridge` [package](https://cran.r-project.org/web/packages/wooldridge/), you might feel like a deer in headlights! (Do fish ever get caught in headlights?) If you want to look for a different dataset and aren't sure where to start, you might try: searching the [documentation PDF](https://cran.r-project.org/web/packages/wooldridge/wooldridge.pdf) for keywords or phrases. If you find a dataset that sounds interesting, you can write new __R__ code using the examples below, but replace `fish` with the title of the dataset you want to look at.

## Reading in the Data

To read in data, we'll use the `data()` function provided by the `wooldridge` package. As the parameter inside `data`, we'll put the dataset name. A full list of the datasets and descriptions of each of the columns can be found [here](https://www.rdocumentation.org/packages/wooldridge/versions/1.3.1).

As an example, let's take a look at the `fish` dataset. Using `data()` loads the dataset into a variable with the same name as the dataset.

In [ ]:
data("fish")

In [ ]:
head(fish)

To get more details about the `fish` dataset, we can use the `?` command to access more info about the data. Let's check it out: 

In [ ]:
?fish

This dataset is from the Fulton Fish Market and has 97 observations with 20 columns. Here is the plot of log price of fish transacted (Y) against log quantity of fish transacted:

In [ ]:
plot(fish$ltotqty, fish$lavgprc)

### Fulton Fish Market

As a primer, let's learn a little more about the paper that this dataset is from. The original paper, Markets: The Fulton Fish Market by Professor Kathryn Graddy [here](https://pubs.aeaweb.org/doi/pdfplus/10.1257/jep.20.2.207). You can also watch a short video below for an explanation by Prof Graddy. 

In [ ]:
install.packages("vembedr")
install.packages("Hmisc")
install.packages("ggplot2")

library("ggplot2")
library("Hmisc")
library("vembedr")
library("dplyr")

Below is a 4-minute video overview by Kathryn Graddy of Brandeis University of her work measuring prices, transactions, ethnicity of buyers, and the mob influence in the Fulton Fish Market of New York City in the early 1990s.

In [ ]:
embed_youtube("5hJF8zNJg5I")

Now that we know a little bit about the context of the dataset, let's take a more in-depth look at some of the data specifics. To do this, we can use the `nrow` and `ncol` commands to get the number of rows and columns

In [ ]:
# Number of rows in the dataset
nrow(fish)

In [ ]:
# Number of columns in the dataset
ncol(fish)

To get a sense about some of the summary statistics of the dataset, let's use the `describe` command.

In [ ]:
describe(fish)

In [ ]:
head(fish)

Are there any values that pop out at you? Remember you can use the `?` command to look at what each of the columns mean.

Note that `fish` is a condensed data set, with only 97 observations in total. Prof. Graddy probably had many more observations in her original dataset, and a quick look at her [1995 RAND J. Econ paper](https://www-jstor-org.libproxy.berkeley.edu/stable/2556036) confirms this.

Let's look at prices and quantity throughout the week. To do this, we'll group by the day of the week and create new columns that are the mean of price and quantity.

In [ ]:
groupby_day <-fish %>%
    group_by(mon, tues, wed, thurs) %>%
    summarize(mean_quantity = mean(totqty),
             mean_price = mean(avgprc))

groupby_day_clean <- groupby_day[c(2:5),c("mean_quantity","mean_price")]

groupby_day_clean$day=c("Thursday","Wednesday","Tuesday","Monday")

groupby_clean_order <- groupby_day_clean[,c("day","mean_quantity","mean_price")]

In [ ]:
groupby_clean_order

Looks like the price and quantity for fish is higher on Thursday.

Here is a regression analysis shown by Wooldridge on p. 572 of the 4th edition of his textbook, and also shown in Table 2 of Graddy's [2006 J Econ Perspect paper](https://pubs.aeaweb.org/doi/pdfplus/10.1257/jep.20.2.207). Consider this approach to modeling the quantity of fish demanded: 

$$
ltotqty_i = \alpha + \beta_p \ lavgprc_i + \beta_m \ mon_i + \beta_t \ tues_i 
+ \beta_w \ wed_i + \beta_{th} \ thurs_i + \epsilon_i
$$

where `ltotqty` is the log total quantity sold (to all customers); `lavgprc` is the log average price; and the rest of the variables are indicators for weekdays in the week. Friday is omitted, and apparently the market was closed on the weekend. The variable of interest is the demand elasticity $\beta_p$, which is the percentage change in quantity demanded for a percentage change in price.

In [ ]:
demand_reg <- lm(ltotqty ~ lavgprc 
                 + mon + tues + wed + thurs, 
                 data = fish)
summary(demand_reg)

The demand elasticity here is $-0.52$, indicating somewhat inelastic demand. In a handy chart you can find in <i>Health Economics</i> by Bhattacharya, Hyde, and Tu, this elasticity is between those for coffee ($-0.25$) and movies ($-0.9$).

You might be concerned about whether shifts in supply or demand are actually creating these effects, however. That is the fundamental problem in examining market data; either supply or demand could be moving, or both could. Graddy was able to shed some light on that question by examining <b>weather data</b>, which she motivates on page 215 of her JEP piece. She argues that theory and evidence imply that weather only affects the market through supply.

## Bonus round

### Instrumental variables estimation

Hold on to your hats. Let us explore a form of instrumental variables estimation called "two stage least squares" using `ivreg` in __R__. <font color="red">You do NOT have to run instrumental variables in your term project.</font> We are doing it here because Graddy did it, Wooldridge suggests this exercise in an advanced chapter, and because we are currently studying instrumental variables in <i>Mastering Metrics</i>.

In [ ]:
install.packages("ivreg", dependencies = TRUE)

In [ ]:
library("ivreg")

Let us follow Wooldridge's advice and estimate the same model from above, but this time with two-stage least squares (2SLS), a very common form of IV estimation:

$$
ltotqty_i = \alpha + \beta_p \ lavgprc_i + \beta_m \ mon_i + \beta_t \ tues_i 
+ \beta_w \ wed_i + \beta_{th} \ thurs_i + \epsilon_i
$$

We will use `wave2` and `wave3`, both of which are measures of recent wave height around NYC, as instrumental variables. The argument is that the log average price is endogenously determined by supply and demand, but that running IV with weather variables will leverage the causality running from weather to supply and then to the price, and thus will reveal the true slope of the demand curve.

(Intuition: the height of the waves around NYC probably doesn't much affect the typical New Yorker's appetite for fish. But it does affect how hard it is to retrieve fish from the ocean.)

The syntax for `ivreg()` is very similar to that of `lm()`, except you need a "pipe" symbol: "|". The pipe appears after your original equation, and it needs to include the exogenous variables plus any instrumental variables.

In [ ]:
demand_ivreg <- ivreg(ltotqty ~ lavgprc 
                      + mon + tues + wed + thurs 
                      | mon + tues + wed + thurs + wave2 + wave3, 
                      data = fish)
summary(demand_ivreg)

What happened to the price elasticity of demand when we ran IV versus ordinary least squares above? Curious? Read Graddy's JEP piece for more.

<div style="text-align: right"> <span style="font-family:Papyrus; ">And they lived happily ever after. The End.</span></div>